In [1]:
# 1. Setup de caminhos locais
import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path

# Adiciona src ao path (notebook está em notebooks/)
sys.path.insert(0, str(Path(".").absolute().parent))

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "14_preprocess_ptbr"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)

BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [2]:
# 2. Carregar dados de entrada (base limpa do notebook 13)
import pandas as pd, os

in_file = os.path.join(INTERIM_PATH, "news_clean_multisource.parquet")
assert os.path.exists(in_file), f"Arquivo não encontrado: {in_file}. Rode o nb 13 primeiro."

df = pd.read_parquet(in_file)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).copy()

print("="*80)
print("BASE LIMPA CARREGADA (pós-ETL)")
print("="*80)
print(f"Arquivo: {in_file}")
print(f"Total de registros: {len(df):,}")
print(f"Dias distintos: {df['date'].nunique():,}")
print(f"Período: {df['date'].min()} → {df['date'].max()}")
print("="*80)
display(df.head(3))

BASE LIMPA CARREGADA (pós-ETL)
Arquivo: C:\Users\ander\OneDrive\TCC_USP\data_interim\news_clean_multisource.parquet
Total de registros: 91,941
Dias distintos: 2,771
Período: 2018-01-02 00:00:00 → 2025-11-19 00:00:00


,date,title,text,source,url,origin
0,2018-01-02,Bovespa começa 2018 no azul apoiada em ações d...,None,g1.globo.com,https://g1.globo.com/economia/noticia/bovespa-...,None
1,2018-01-02,"Com alta de 504 % em 2017 , Magazine Luiza ent...",None,jj.com.br,http://www.jj.com.br/noticias-52569-com-alta-d...,None
2,2018-01-02,Ibovespa estica rali e abre 2018 com pontuação...,None,tribunahoje.com,http://tribunahoje.com/noticias/economia/2018/...,None


In [3]:
# Dependência já deve estar instalada via requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 3.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ spaCy carregado: core_news_lg 3.8.0


In [3]:
# 3. Configurações de NLP
import nltk
from nltk.corpus import stopwords
from unidecode import unidecode

nltk.download("stopwords", quiet=True)
STOP_PT = set(stopwords.words("portuguese"))
USE_SPACY = True
try:
    import spacy
    nlp = spacy.load("pt_core_news_lg")
    print("[INFO] Pré-processamento PT-BR usando spaCy pt_core_news_lg (lematização completa)")
except Exception as exc:
    USE_SPACY = False
    nlp = None
    print(f"[INFO] Pré-processamento PT-BR usando fallback NLTK (sem lematização)")
    print(f"[AVISO] spaCy indisponível: {exc}")

[INFO] Pré-processamento PT-BR usando fallback NLTK (sem lematização)
[AVISO] spaCy indisponível: [E050] Can't find model 'pt_core_news_lg'. It doesn't seem to be a Python package or a valid path to a data directory.


In [4]:
# 4. Utilitários de limpeza PT-BR
import re

URL_RE   = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w{2,}\b")
NUM_RE   = re.compile(r"\b\d+(?:[.,]\d+)?\b")
WS_RE    = re.compile(r"\s+")

def normalize_whitespace(t: str) -> str:
    return WS_RE.sub(" ", t).strip()

def build_text(row):
    t = (str(row.get("title", "")) + " " + str(row.get("text", ""))).strip()
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    # preserva números? para sentimento, em geral remove:
    t = NUM_RE.sub(" ", t)
    return normalize_whitespace(t)

In [5]:
# 5. Funções de pré-processamento (spaCy ou fallback)
def preprocess_spacy_pt(text: str):
    doc = nlp(text)
    lemmas = []
    for tok in doc:
        if tok.is_space or tok.is_punct or tok.like_url or tok.like_email:
            continue
        if tok.is_stop:
            continue
        lemma = tok.lemma_.lower().strip()
        if not lemma or len(lemma) < 2:
            continue
        # remove só pontuação residual
        if re.fullmatch(r"[^\wáàâãéêíóôõúç]+", lemma, flags=re.IGNORECASE):
            continue
        lemmas.append(lemma)
    clean = " ".join(lemmas)
    return clean, lemmas

def preprocess_fallback_pt(text: str):
    # regex + NLTK + unidecode (sem lematização)
    tx = text.lower()
    tx = URL_RE.sub(" ", tx)
    tx = EMAIL_RE.sub(" ", tx)
    tx = re.sub(r"[^a-zà-úç\s]", " ", tx, flags=re.IGNORECASE)
    tx = normalize_whitespace(tx)
    tokens = [w for w in tx.split() if w not in STOP_PT and len(w) > 1]
    clean = " ".join(tokens)
    return clean, tokens

In [6]:
# 6. Aplicar pré-processamento
from tqdm.auto import tqdm
tqdm.pandas()

df = df.assign(text_full=df.apply(build_text, axis=1))

if USE_SPACY:
    out = df["text_full"].progress_apply(preprocess_spacy_pt)
else:
    out = df["text_full"].progress_apply(preprocess_fallback_pt)

df["clean_text"] = out.apply(lambda x: x[0])
df["lemmas"]     = out.apply(lambda x: x[1])
df["n_tokens"]   = df["lemmas"].apply(len)

# Campos auxiliares
df["day"] = df["date"].dt.floor("D")
print("Pré-processamento concluído. Exemplo:")
display(df[["date","source","title","clean_text","n_tokens"]].head(5))

  0%|          | 0/91941 [00:00<?, ?it/s]

Pré-processamento concluído. Exemplo:


,date,source,title,clean_text,n_tokens
0,2018-01-02,g1.globo.com,Bovespa começa 2018 no azul apoiada em ações d...,bovespa começa azul apoiada ações blue chips none,8
1,2018-01-02,jj.com.br,"Com alta de 504 % em 2017 , Magazine Luiza ent...",alta magazine luiza entra carteira ibovespa none,7
2,2018-01-02,tribunahoje.com,Ibovespa estica rali e abre 2018 com pontuação...,ibovespa estica rali abre pontuação recorde fe...,9
3,2018-01-02,portalodia.com,"Com alta de 504 % em 2017 , Magazine Luiza ent...",alta magazine luiza entra carteira ibovespa none,7
4,2018-01-02,correiodoestado.com.br,"Com alta de 504 % em 2017 , Magazine Luiza ent...",alta magazine luiza entra carteira ibovespa none,7


In [7]:
# 7. Salvar dataset limpo + BOW diário por fonte + VALIDAÇÃO TCC
import numpy as np

clean_parquet = os.path.join(PROC_PATH, "news_clean.parquet")
clean_csv     = os.path.join(PROC_PATH, "news_clean.csv")

# dataset limpo
cols_out = ["date","day","source","url","origin","title","text","clean_text","lemmas","n_tokens"]
df_out = df[cols_out].copy()

# ========== VALIDAÇÃO OBRIGATÓRIA TCC ==========
MIN_DAYS_THRESHOLD = 200
n_days_final = df_out["day"].nunique()
n_records_final = len(df_out)
date_min = df_out["day"].min()
date_max = df_out["day"].max()

print("\n" + "="*80)
print("VALIDAÇÃO DA BASE PROCESSADA (TCC)")
print("="*80)
print(f"Total de registros: {n_records_final:,}")
print(f"Dias distintos: {n_days_final:,}")
print(f"Período: {date_min} → {date_max}")
print(f"Tokens médios/doc: {df_out['n_tokens'].mean():.1f}")

if n_days_final < MIN_DAYS_THRESHOLD:
    raise RuntimeError(
        f"❌ BASE INSUFICIENTE PARA TCC!\n"
        f"   Dias encontrados: {n_days_final}\n"
        f"   Mínimo exigido: {MIN_DAYS_THRESHOLD}\n"
        f"   Verifique notebooks anteriores (12, 13)."
    )

print(f"\n✅ VALIDAÇÃO APROVADA: {n_days_final:,} dias >= {MIN_DAYS_THRESHOLD}")
print("   STATUS: Base adequada para TCC")
print("="*80 + "\n")
# ================================================

df_out.to_parquet(clean_parquet, index=False)
df_out.to_csv(clean_csv, index=False, encoding="utf-8")
print("✅ Salvo dataset limpo:\n  ", clean_parquet, "\n  ", clean_csv)

# BOW diário por fonte (para TF-IDF no nb 15)
bow = (
    df[["day","source","lemmas"]]
    .explode("lemmas")
    .dropna(subset=["lemmas"])
    .groupby(["day","source","lemmas"], as_index=False)
    .size()
    .rename(columns={"size":"tf"})
)

bow_file = os.path.join(PROC_PATH, "bow_daily.parquet")
bow.to_parquet(bow_file, index=False)
print("✅ Salvo BOW diário por fonte:\n  ", bow_file)
display(bow.head(10))


VALIDAÇÃO DA BASE PROCESSADA (TCC)
Total de registros: 91,941
Dias distintos: 2,771
Período: 2018-01-02 00:00:00 → 2025-11-19 00:00:00
Tokens médios/doc: 8.6

✅ VALIDAÇÃO APROVADA: 2,771 dias >= 200
   STATUS: Base adequada para TCC

✅ Salvo dataset limpo:
   C:\Users\ander\OneDrive\TCC_USP\data_processed\news_clean.parquet 
   C:\Users\ander\OneDrive\TCC_USP\data_processed\news_clean.csv
✅ Salvo dataset limpo:
   C:\Users\ander\OneDrive\TCC_USP\data_processed\news_clean.parquet 
   C:\Users\ander\OneDrive\TCC_USP\data_processed\news_clean.csv
✅ Salvo BOW diário por fonte:
   C:\Users\ander\OneDrive\TCC_USP\data_processed\bow_daily.parquet
✅ Salvo BOW diário por fonte:
   C:\Users\ander\OneDrive\TCC_USP\data_processed\bow_daily.parquet


,day,source,lemmas,tf
0,2018-01-02,bemparana.com.br,alta,1
1,2018-01-02,bemparana.com.br,ações,3
2,2018-01-02,bemparana.com.br,bate,1
3,2018-01-02,bemparana.com.br,boeing,3
4,2018-01-02,bemparana.com.br,bolsa,1
5,2018-01-02,bemparana.com.br,brasileira,1
6,2018-01-02,bemparana.com.br,divisão,3
7,2018-01-02,bemparana.com.br,embraer,3
8,2018-01-02,bemparana.com.br,emenda,1
9,2018-01-02,bemparana.com.br,financeiro,1


In [8]:
# 8. QC rápido (distribuições e amostras)
print("Tamanho do corpus (docs):", df.shape[0])
print("Tokens por doc (p50/p90):",
      int(df["n_tokens"].median()), "/", int(df["n_tokens"].quantile(0.9)))

print("\nTop 15 tokens por origem (amostra):")
top = (
    df[["origin","lemmas"]].explode("lemmas")
      .groupby(["origin","lemmas"]).size()
      .reset_index(name="tf").sort_values(["origin","tf"], ascending=[True, False])
)
for org in top["origin"].unique()[:3]:  # mostra até 3 origens
    subt = top[top["origin"]==org].head(15)
    print(f"\n{org}:")
    print(subt[["lemmas","tf"]].to_string(index=False))
print("\nFeito ✅")

Tamanho do corpus (docs): 91941
Tokens por doc (p50/p90): 8 / 11

Top 15 tokens por origem (amostra):

None:
  lemmas    tf
    none 91941
ibovespa 31158
   dólar 20809
   fecha 15621
    alta 13492
   bolsa 12994
     aes 11245
    sobe 10180
   queda  9971
    após  9799
     cai  9268
    hoje  8343
  pontos  7774
   preço  7634
   saiba  7387

Feito ✅

None:
  lemmas    tf
    none 91941
ibovespa 31158
   dólar 20809
   fecha 15621
    alta 13492
   bolsa 12994
     aes 11245
    sobe 10180
   queda  9971
    após  9799
     cai  9268
    hoje  8343
  pontos  7774
   preço  7634
   saiba  7387

Feito ✅
